In [ ]:
import numpy as np
import math
import scipy
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import shapely
import pylatex
import colormaps as cmaps
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm
from pyproj import Proj, transform
import webbrowser
import folium
from folium import GeoJson, GeoJsonTooltip

ox.settings.log_console = True

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib_scalebar.scalebar import ScaleBar
#matplotlib.verbose.level = 'debug-annoying'
plt.rcParams.update({
    "text.usetex": True,
    "font.size": 16,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.dpi": 600,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral"
})

bgCol = "#212121ff"
bgdy = '#940B13'
lred = '#D10D18'

cmap = mpl.cm.viridis
cmap_reversed = mpl.colormaps['viridis_r']
greys = mpl.cm.Greys
greens = mpl.cm.Greens
oranges = mpl.cm.Oranges
blues = mpl.cm.Blues
purples = mpl.cm.Purples
clwm = mpl.cm.coolwarm
brbg = mpl.cm.BrBG_r

# my_dpi=96
# w=8
# h=5

In [ ]:
### DEFINE CONSTANTS ###
tol = 58.6
no2limAQ = 200 # in ug/m3, as per The Air Quality Standards Regulations 2010
no2traffic = 0.68 # proportion of roadside NO2 caused by motor vehicle traffic
ppb_to_ugm3 = 1.9387 # multiply NO2 in ppb to get ug/m3

suffixUE = '_out'
suffixPO = '_out'

mainDir = '' # INSERT WORKING DIRECTORY
imgOutDir = mainDir + 'plots/'

# outputs from TA optimisation
flowUE_path = mainDir + 'outputs/UE_flow' + suffixUE + '.csv'
flowPO_path = mainDir + 'outputs/PO_flow' + suffixPO + '.csv'
no2UE_path = mainDir + 'outputs/UE_NO2' + suffixUE + '.csv'
no2PO_path = mainDir + 'outputs/PO_NO2' + suffixPO + '.csv'
no2UE_ugm3_path = mainDir + 'outputs/UE_NO2_ugm3' + suffixUE + '.csv'
no2PO_ugm3_path = mainDir + 'outputs/PO_NO2_ugm3' + suffixPO + '.csv'
no2UE_Tot_path = mainDir + 'outputs/UE_NO2_Tot' + suffixUE + '.csv'
no2PO_Tot_path = mainDir + 'outputs/PO_NO2_Tot' + suffixPO + '.csv'
no2UE_Prop_path = mainDir + 'outputs/UE_NO2_Prop' + suffixUE + '.csv'
no2PO_Prop_path = mainDir + 'outputs/PO_NO2_Prop' + suffixPO + '.csv'
densityUE_path = mainDir + 'outputs/UE_density' + suffixUE + '.csv'
densityPO_path = mainDir + 'outputs/PO_density' + suffixPO + '.csv'
TT_UE_path = mainDir + 'outputs/UE_TT' + suffixUE + '.csv'
TT_PO_path = mainDir + 'outputs/PO_TT' + suffixPO + '.csv'
intInd_path = mainDir + 'outputs/interventionIndex' + suffixUE + '.csv'

# OD demand files
od_names_path = mainDir + 'inputs/OD_names.csv'
od_mat_path = mainDir + 'inputs/OD_mat.csv'
od_list_path = mainDir + 'inputs/OD_list_tol' + str(tol) + '.csv'
demand_path = mainDir + 'inputs/demand.csv'
solo_nodes_path = mainDir + 'inputs/soloNodes_tol' + str(tol) + '.csv'

# shape data
lsoa_path = mainDir + 'inputs/LSOA_2011_EW_BGC.zip' # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD SHAPE FILES FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
msoa_path = mainDir + 'inputs/MSOA_2011_EW_BGC.zip'
lsoa_msoa_lookup_path = mainDir + 'inputs/LSOA_to_MSOA_2011.csv' # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD LOOKUP TABLE FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
sheff_path = mainDir + 'inputs/City_Boundary.shp'

# graph inputs (reduced network)
edge_path = mainDir + 'inputs/edges.csv'
node_path = mainDir + 'inputs/nodes.csv'
lsoa_dists_path = mainDir + 'inputs/lDists_tol58.6.csv'

# lookups
lsoa_msoa_lookup_21_path = mainDir + 'inputs/LSOA_to_MSOA_2021.csv' # TOO LARGE TO UPLOAD TO GITHUB - DOWNLOAD LOOKUP TABLE FROM OPEN GEOGRAPHY PORTAL (OPEN ACCESS)
lsoa_11_21_path = mainDir + 'inputs/LSOA_2011_to_LSOA_2021.csv'
 # ONS annual income data (2020)
income_path = mainDir + 'inputs/total_income_2020.csv' # total
income_net_path = mainDir + 'inputs/net_income_2020.csv' # net

# size of LSOAs (area)
OA_areas_path = mainDir + 'inputs/SAM_LSOA_DEC_2021_EW_in_KM.csv'

# population and company statistics
company_house_path = mainDir + 'inputs/company_house_table_sheffield.csv'
population_density_path = mainDir + 'inputs/lsoa_population_density_mid2022.csv'

# validation #

# SCC sensor data (flow)
flow_path = mainDir + 'inputs/Flow_Sensors_SUFO_2019.csv'

# NAME OF OUTPUT FILE OF THIS SCRIPT
shps_path = mainDir + 'outputs/results_LSOA_aggregated.csv'

In [ ]:
### LOAD DATA ###
# names of ODs
ODs = pd.read_csv(od_names_path, header=None)
ODs = ODs.iloc[1:].reset_index(drop=True)
demand = pd.read_csv(demand_path, header=None)
demand = demand.iloc[1:].reset_index(drop=True)
soloNodes = pd.read_csv(solo_nodes_path)

# TA optimisation outputs
# import flow results
flowUE = pd.read_csv(flowUE_path, header=None)
flowPO = pd.read_csv(flowPO_path, header=None)
# import NO2 results
no2UE = pd.read_csv(no2UE_ugm3_path, header=None)
no2PO = pd.read_csv(no2PO_ugm3_path, header=None)
# import density results
densityUE = pd.read_csv(densityUE_path, header=None)
densityPO = pd.read_csv(densityPO_path, header=None)
# import travel times
TT_UE = pd.read_csv(TT_UE_path, header=None)
TT_PO = pd.read_csv(TT_PO_path, header=None)
# intervention index
intInd = pd.read_csv(intInd_path, header=None)

# import demand matrix and demand sums per LSOA
ODmat = pd.read_csv(od_mat_path, header=None)
ODmat = ODmat.iloc[1:,:].reset_index(drop=True)

# OD list
ODlist = pd.read_csv(od_list_path, header=None)
ODlist = ODlist.iloc[1:,:].reset_index(drop=True)

# LSOA to MSOA lookup tables
lsoa_msoa_lookup_21 = pd.read_csv(lsoa_msoa_lookup_21_path)
lsoa_11_21 = pd.read_csv(lsoa_11_21_path)
lsoa_msoa_lookup = pd.read_csv(lsoa_msoa_lookup_path)

# OA areas lookup table
OA_areas = pd.read_csv(OA_areas_path)

# ONS annual income data (2020)
income = pd.read_csv(income_path)
income_net = pd.read_csv(income_net_path)

# population and company statistics
compHouse = pd.read_csv(company_house_path)
popDens = pd.read_csv(population_density_path)

# validation #

# SCC sensor data (flow)
flow = pd.read_csv(flow_path)

# spatial data
lsoas = gpd.read_file(lsoa_path).to_crs('WGS84')
msoas = gpd.read_file(msoa_path).to_crs('WGS84')

sheff = gpd.read_file(sheff_path)
sheff = sheff.to_crs('WGS84')

# import node/LSOA relations
G_edges = gpd.read_file(edge_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
G_nodes = pd.read_csv(node_path)#, header=None)
G_nodes = G_nodes[['osmid','x','y','LSOA','LSOA_Lon','LSOA_Lat']]
lDists = gpd.read_file(lsoa_dists_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")

# reduced network
G2 = ox.graph_from_gdfs(G_nodes.set_index('osmid'), G_edges.set_index(['u','v','key']), graph_attrs={'crs':'WGS84'})

In [ ]:
def ENtoLL(easting,northing):
    lat,lon = transform(Proj('epsg:27700'), Proj('epsg:4326'), easting, northing)
    return lon,lat

In [ ]:
### IF SCRIPT HAS BEEN RUN IN FULL BEFORE ###
shps = gpd.read_file(shps_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
shps = shps.set_crs('WGS84')

In [ ]:
flow['geometry'] = [shapely.Point(xy) for xy in zip(flow.Lon, flow.Lat)]
flow = gpd.GeoDataFrame(flow, crs="WGS84", geometry=flow['geometry'])

In [ ]:
no2UE = no2UE * 9.034974
no2PO = no2PO * 9.034974

In [ ]:
G_edges = G_edges.assign(flowPO = flowPO, flowUE = flowUE, no2PO = no2PO, no2UE = no2UE)

In [ ]:
# G_edges['no2UE'] = G_edges['no2UE'] * 9.034974
# G_edges['no2PO'] = G_edges['no2PO'] * 9.034974

In [ ]:
xy = sheff.get_coordinates()

In [ ]:
km = np.array(G_edges['length']).astype('float')/1000

# get flow per km of road
flowUE_km = flowUE.div(km, axis=0)
flowPO_km = flowPO.div(km, axis=0)

# get NO2 per km of road
no2UE_km = no2UE.div(km, axis=0)
no2PO_km = no2PO.div(km, axis=0)

In [ ]:
msoas = msoas.loc[msoas['MSOA11NMW'].str.contains('Sheffield')].reset_index(drop=True)

In [ ]:
shps = lsoas[lsoas.LSOA11NM.str.contains('Sheffield')].reset_index(drop=True)

In [ ]:
diagDemand = np.array(ODmat).diagonal()

rowSums = np.array(ODmat.sum(axis=0))
colSums = np.array(ODmat.sum(axis=1))

totalDemand = rowSums + colSums - diagDemand
interDemand = totalDemand - diagDemand

ODs = ODs.assign(Demand_O = rowSums, Demand_D = colSums, Demand_Total = totalDemand,
                 Intra_LSOA = diagDemand, Inter_LSOA = interDemand)

In [ ]:
# correlation between inter- and intra- LSOA demand
np.corrcoef(diagDemand, interDemand)

In [ ]:
shps = shps.join(ODs.set_index(0), on='LSOA11CD')

In [ ]:
# Outputs aggregated by LSOA
shps['flowUE'] = 0
shps['flowPO'] = 0

shps['no2UE'] = 0
shps['no2PO'] = 0

# intervention index
shps['intInd'] = 0

# road count
shps['rdCount'] = 0

In [ ]:
no2s = pd.concat([no2UE, no2PO], axis=1).rename(columns={0: 'no2UE', 1: 'no2PO'})
newColsList = list(shps.LSOA11CD)
# one-hot encode the LSOAs
roads = pd.DataFrame(columns=newColsList, index=no2s.index)

In [ ]:
# get single node LSOAs
inds = np.where(shps['LSOA11CD'].isin(soloNodes['LSOA']))[0]
len(inds)

In [ ]:
lDists.NodeID = lDists.NodeID.astype('float').astype('int')

In [ ]:
%%capture

for index,row in G_edges.iterrows():
    n0 = int(row.iloc[0]) # start node
    n1 = int(row.iloc[1]) # end node
    l0 = lDists['LSOA'].loc[lDists.NodeID==n0].reset_index().LSOA # lsoa of n0
    l1 = lDists['LSOA'].loc[lDists.NodeID==n1].reset_index().LSOA # lsoa of n1
    geom = row['geometry'] # linestring of road
    coords = shapely.get_coordinates(geom) # individual nodes of linestring
    matches = np.zeros([len(shps),1])
    for lInd, lsoa in shps.iterrows():
        polygon = lsoa.geometry
        intersect = polygon.intersection(geom) # does the lsoa intersect the linestring?
        if not shapely.is_empty(intersect):
            matches[lInd] = 1 # if there is an intersection, count the lsoa as a match
    hits = np.where(matches)[0] # indices of 'hit' lsoas
    hits = np.append(hits, np.where((shps['LSOA11CD'].isin(l0)) | (shps['LSOA11CD'].isin(l1)))[0]) # if the node has been nearest joined to another lsoa for OD assignment, include these lsoas too
    hits = np.unique(hits) # in case of duplicates

    shps.loc[hits, 'flowUE'] = shps.loc[hits, 'flowUE'] + flowUE.iloc[index][0]
    shps.loc[hits, 'flowPO'] = shps.loc[hits, 'flowPO'] + flowPO.iloc[index][0]

    shps.loc[hits, 'no2UE'] = shps.loc[hits, 'no2UE'] + no2UE.iloc[index][0]
    shps.loc[hits, 'no2PO'] = shps.loc[hits, 'no2PO'] + no2PO.iloc[index][0]

    shps.loc[hits, 'intInd'] = shps.loc[hits, 'intInd'] + intInd.iloc[index][0]

    shps.loc[hits, 'rdCount'] = shps.loc[hits, 'rdCount'] + 1

    # assign LSOA(s) to the roads
    roads.loc[index, shps.loc[hits, 'LSOA11CD'].values] = 1
    

In [ ]:
roads = roads.fillna(0)

In [ ]:
shps = shps.assign(flowUE_mean=0, flowUE_median=0, flowPO_mean=0, flowPO_median=0,
                    no2UE_mean=0, no2UE_median=0, no2PO_mean=0, no2PO_median=0)

In [ ]:
%%capture

for index, row in shps.iterrows():
    lsoa = row['LSOA11CD']
    vals = roads[lsoa]
    hits = np.where(vals!=0)
    # flows
    shps.loc[index, 'flowUE_mean'] = np.mean(flowUE.iloc[hits])
    shps.loc[index, 'flowUE_median'] = np.median(flowUE.iloc[hits])
    shps.loc[index, 'flowPO_mean'] = np.mean(flowPO.iloc[hits])
    shps.loc[index, 'flowPO_median'] = np.median(flowPO.iloc[hits])
    # NO2
    shps.loc[index, 'no2UE_mean'] = np.mean(no2UE.iloc[hits])
    shps.loc[index, 'no2UE_median'] = np.median(no2UE.iloc[hits])
    shps.loc[index, 'no2PO_mean'] = np.mean(no2PO.iloc[hits])
    shps.loc[index, 'no2PO_median'] = np.median(no2PO.iloc[hits])

In [ ]:
shps = shps.assign(no2UE_meanByRdCount = shps['no2UE']/shps['rdCount'],
                   no2PO_meanByRdCount = shps['no2PO']/shps['rdCount'])

In [ ]:
G_edges['criticalDensity'] = G_edges['criticalDensity'].astype('float')
G_edges['capacity'] = G_edges['capacity'].astype('float')

In [ ]:
print(np.mean(G_edges['criticalDensity']), np.mean(G_edges['capacity']))

In [ ]:
display(lsoa_11_21.head())

In [ ]:
lsoa_msoa_lookup_21 = lsoa_msoa_lookup_21[['pcds','lsoa21cd','msoa21cd','ladnm']]

compHouse = compHouse.set_index('RegAddress.PostCode').join(lsoa_msoa_lookup_21.set_index('pcds'))

compHouse = compHouse.merge(lsoa_11_21, how="left", left_on="lsoa21cd", right_on="LSOA21CD")

compHouse_LSOA = compHouse.groupby('LSOA11CD')

groupSizes = compHouse_LSOA.size()
groupSizes = groupSizes.sort_values(ascending=False)
groupSizes = pd.DataFrame(groupSizes).rename(columns={0:"company_count"})

In [ ]:
shps = shps.merge(groupSizes, how='left', left_on='LSOA11CD', right_on='LSOA11CD')

In [ ]:
OA_areas = OA_areas[['LSOA21CD','Extent of the Realm (Area in KM2)']].rename(columns={'Extent of the Realm (Area in KM2)':'Area'})
shps = shps.merge(OA_areas, how='left', left_on='LSOA11CD', right_on='LSOA21CD')

popDens = popDens[['LSOA 2021 Code','Total']].rename(columns={'Total':'population'})
shps = shps.merge(popDens, how='left', left_on='LSOA11CD', right_on='LSOA 2021 Code')

In [ ]:
shps = shps.assign(popn = np.nan)
for index,row in shps.iterrows():
    val = shps.loc[shps.index==index,'population'][index]
    #val = shps.loc[shps.index==index,'population'][index].replace(",", "")
    if not pd.isnull(val):
        val = val.replace(",", "")
    shps.loc[shps.index==index,'popn'] = float(val)

In [ ]:
shps['flowDiff'] = shps.flowPO - shps.flowUE
shps['no2Diff'] = shps.no2PO - shps.no2UE

shps['flowDiff_mean'] = shps.flowPO_mean - shps.flowUE_mean
shps['no2Diff_mean'] = shps.no2PO_mean - shps.no2UE_mean

shps['flowDiff_median'] = shps.flowPO_median - shps.flowUE_median
shps['no2Diff_median'] = shps.no2PO_median - shps.no2UE_median

In [ ]:
shps['no2Diff_cat'] = pd.cut(shps['no2Diff'],
                            bins=[float('-Inf'), 0, float('Inf')],
                            labels=['Reduced', 'Increased'])

shps['no2Diff_mean_cat'] = pd.cut(shps['no2Diff_mean'],
                            bins=[float('-Inf'), 0, float('Inf')],
                            labels=['Reduced', 'Increased'])

shps['no2Diff_median_cat'] = pd.cut(shps['no2Diff_median'],
                            bins=[float('-Inf'), 0, float('Inf')],
                            labels=['Reduced', 'Increased'])

In [ ]:
shps[['no2Diff','no2Diff_cat','no2Diff_mean_cat','no2Diff_median_cat']]

In [ ]:
print(sum(shps.loc[~np.isnan(shps['no2UE']), 'no2UE']), sum(shps.loc[~np.isnan(shps['no2PO']), 'no2PO']))

In [ ]:
income = income.assign(Income_Total_2020 = 0)
for index,row in income.iterrows():
    val = income.loc[index, 'Total annual income (£)'].replace(",", "")
    income.loc[index, 'Income_Total_2020'] = float(val)

In [ ]:
income_net = income_net.assign(Income_Total_2020 = 0)
for index,row in income_net.iterrows():
    val = income_net.loc[index, 'Net annual income (£)'].replace(",", "")
    income_net.loc[index, 'Income_Net_2020'] = float(val)

In [ ]:
# percentage of MSOAs accounted for in income data
print(len(np.where(income['MSOA code'].isin(msoas['MSOA11CD']))[0])*100/len(msoas))

In [ ]:
msoas = msoas.merge(income[['MSOA code','Income_Total_2020']], how='left', left_on='MSOA11CD', right_on='MSOA code')
msoas = msoas.merge(income_net[['MSOA code','Income_Net_2020']], how='left', left_on='MSOA11CD', right_on='MSOA code')

In [ ]:
# correlation of company count on total demand
np.corrcoef(shps['company_count'], totalDemand)

In [ ]:
# correlation of company count and population
np.corrcoef(shps.loc[~np.isnan(shps['popn']),'company_count'], shps.loc[~np.isnan(shps['popn']),'popn'])

In [ ]:
shps = shps.assign(intIndMean = np.divide(shps['intInd'], shps['rdCount']))

In [ ]:
#shps.to_csv(shps_path)

In [ ]:
# JOIN NEAREST ROAD TO SCC FLOW SENSORS #

In [ ]:
flowUEval = pd.DataFrame(flow['FlowHour'])
flowUEval = flowUEval.assign(FlowModel = 0)

In [ ]:
%%capture

for index, row in flow.iterrows():
    pnt = row['geometry']
    dists = np.zeros([len(G_edges),1]) + 100000
    for edgeIndex, edgeRow in G_edges.iterrows():
        line = G_edges.loc[edgeIndex,'geometry']
        dists[edgeIndex] = pnt.distance(line) # shapely.line_locate_point(line, pnt)
    flowUEval.loc[index, 'FlowModel'] = np.mean(flowUE[dists==min(dists)])

In [ ]:
flowUEval = flowUEval.assign(Factor = np.divide(flowUEval['FlowHour'], flowUEval['FlowModel']/9.674))
flowUEval.loc[np.isinf(flowUEval['Factor']), 'Factor'] = flowUEval['FlowHour']

In [ ]:
#flowUEval.to_csv(mainDir + 'flows_SCC2023_and_modelled2019_postCorrection.csv')

In [ ]:
print(np.mean(flowUEval['Factor']), np.median(flowUEval['Factor']))

In [ ]:
# PLOTTING #

In [ ]:
# Read & simplify South Yorkshire road network
cf = '["highway"~"motorway|trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link|unclassified"]' # CAR NETWORK FOR PAPER
G = ox.graph_from_place('Sheffield', custom_filter=cf, simplify=True)
remove = [node for node, degree in dict(G.degree()).items() if degree < 2]
G.remove_nodes_from(remove)

In [ ]:
edges = ox.graph_to_gdfs(G, nodes=False)

In [ ]:
thicknesses = [None] * len(edges)
lenCheck = [None] * len(edges)
for ii in range(len(edges)):
    lenCheck[ii] = len(edges['highway'].iloc[ii])
    if edges['highway'].iloc[ii] == 'motorway':
        thicknesses[ii] = 1.25
    if edges['highway'].iloc[ii] == 'motorway_link':
        thicknesses[ii] = 1.25
    if edges['highway'].iloc[ii] == 'primary':
        thicknesses[ii] = 1
    if edges['highway'].iloc[ii] == 'primary_link':
        thicknesses[ii] = 1
    if edges['highway'].iloc[ii] == 'secondary':
        thicknesses[ii] = 0.75
    if edges['highway'].iloc[ii] == 'secondary_link':
        thicknesses[ii] = 0.75
    if edges['highway'].iloc[ii] == 'trunk':
        thicknesses[ii] = 0.5
    if edges['highway'].iloc[ii] == 'trunk_link':
        thicknesses[ii] = 0.5
    if edges['highway'].iloc[ii] == 'unclassified':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == 'tertiary':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == 'tertiary_link':
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == ['unclassified', 'tertiary']:
        thicknesses[ii] = 0.1
    if edges['highway'].iloc[ii] == ['tertiary', 'unclassified']:
        thicknesses[ii] = 0.1

In [ ]:
# Plot NETWORK USED FOR MODELLING graph for paper
fig, ax = plt.subplots()
sheff.boundary.plot(ax=ax, color="#4A4949", linewidth=0.3)
sheff.plot(ax=ax, color="#F2F1F1") #2C2C2C ##E7E7E7
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)

# Add a scale bar
scalebar = ScaleBar(
    dx=1,                  # 1 map unit = 1 metre (if CRS is in metres)
    units="km",             
    length_fraction=0.25,  # how wide the scale bar is relative to the map
    location="lower left" # placement
)
ax.add_artist(scalebar)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_carNet.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# PLOT LSOA MAP WITH MARKERS #
# Convert to WGS84 (for web mapping)
gdf_wgs84 = shps.to_crs(epsg=4326)  # EPSG:4326 = WGS84

# Create map centered on Sheffield
m = folium.Map(location=[53.3811, -1.4701], zoom_start=11)

# Add LSOA polygons
GeoJson(
    gdf_wgs84,
    tooltip=GeoJsonTooltip(fields=['LSOA11CD'], aliases=['LSOA Code:'])
).add_to(m)

# Add red dot for Sheffield train station
folium.CircleMarker(
    location=[53.3785, -1.4614],
    radius=6,
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.9,
    popup='Sheffield Train Station'
).add_to(m)

# Add blue dot for Meadowhall shopping centre
folium.CircleMarker(
    location=[53.4142, -1.4121],
    radius=6,
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.9,
    popup='Meadowhall Shopping Centre'
).add_to(m)

# Add green dot for Northern General Hospital
folium.CircleMarker(
    location=[53.4105, -1.4574],
    radius=6,
    color='green',
    fill=True,
    fill_color='green',
    fill_opacity=0.9,
    popup='Northern General Hospital'
).add_to(m)

# Add green dot for Royal Hallamshire Hospital
folium.CircleMarker(
    location=[53.3784, -1.4928],
    radius=6,
    color='green',
    fill=True,
    fill_color='green',
    fill_opacity=0.9,
    popup='Royal Hallamshire Hospital'
).add_to(m)

# Show map
m

#m.save(imgOutDir + 'Sheffield_LSOA_Map_with_Markers.html')

In [ ]:
# view map from file #
webbrowser.open(imgOutDir + 'Sheffield_LSOA_Map_with_Markers.html')

In [ ]:
np.mean(flowUE)

In [ ]:
# demandFactor = 9.67
# G_edges['flowUE'] = np.multiply(G_edges['flowUE'],demandFactor)
# G_edges['flowPO'] = np.multiply(G_edges['flowPO'],demandFactor)
# G_edges['no2UE'] = np.multiply(G_edges['no2UE'],demandFactor)
# G_edges['no2PO'] = np.multiply(G_edges['no2PO'],demandFactor)

In [ ]:
G_edges = G_edges.assign(UE_flow_cap_ratio = G_edges['flowUE']/G_edges['capacity'],
                         PO_flow_cap_ratio = G_edges['flowPO']/G_edges['capacity'])

In [ ]:
vMin = min(min(G_edges['flowUE']), min(G_edges['flowPO']))
vMax = max(max(G_edges['flowUE']), max(G_edges['flowPO']))

In [ ]:
print(max(G_edges['flowUE']), max(G_edges['flowPO']))

In [ ]:
# Plot ROAD-WISE UE flow
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="flowUE",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Traffic flow (veh/hour)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_UE_Flow.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot ROAD-WISE PO flow
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="flowPO",
                categorical=False,
                legend=True,
                #legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (} \mu \textrm{g/m} ^3 \textrm{)}$"},
                legend_kwds={"label": r"$\textrm{Traffic flow (veh/hour)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin=0,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_PO_Flow.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# G_edges['no2UE'] = G_edges['no2UE'] - 29.5281
# G_edges['no2PO'] = G_edges['no2PO'] - 29.5281

In [ ]:
vMin = min(min(G_edges['no2UE']), min(G_edges['no2PO']))
vMax = max(max(G_edges['no2UE']), max(G_edges['no2PO']))

In [ ]:
# Plot ROAD-WISE UE NO2 (scaled up)
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="no2UE",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                ax=ax,
                zorder=1,
                vmin=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_UE_NO2_SCALED.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot ROAD-WISE PO NO2 (scaled up)
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="no2PO",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                ax=ax,
                zorder=1,
                vmin=0,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_PO_NO2_SCALED.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot ROAD-WISE UE flow to capacity ratio
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="UE_flow_cap_ratio",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Traffic flow to capacity ratio}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_UE_Flow_Cap_Ratio.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot ROAD-WISE PO flow to capacity ratio
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#2C2C2C", zorder=0) #2C2C2C
G_edges.plot(column="PO_flow_cap_ratio",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Traffic flow to capacity ratio}$"},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin=0)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor=None,
    edge_color=None,
    edge_linewidth=0.00001,
    node_size=0
)
sheff.boundary.plot(color="black", linewidth=0.5, ax=ax, zorder=2)
#shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Roadwise_PO_Flow_Cap_Ratio.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = 0
vMax = max(max(shps.Demand_O), max(shps.Demand_D))

In [ ]:
# Plot LSOAs coloured by origin demand
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="Demand_O",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Demand (veh/hour)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_Origin_Demand.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by destination demand
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="Demand_D",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Demand (veh/hour)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_Destination_Demand.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by total demand
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="Demand_Total",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Demand (veh/hour)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                vmin=0,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_Total_Demand.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by intra-LSOA demand
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="Intra_LSOA",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Demand (veh/hour)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                vmin=0,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_Intra_LSOA_Demand.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by inter-LSOA demand
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="Inter_LSOA",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Demand (veh/hour)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                vmin=0,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Sheffield_Inter_LSOA_Demand.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = 0
vMax = max(max(shps['flowUE_mean']), max(shps['flowPO_mean']))

In [ ]:
# Plot LSOAs coloured by mean UE flows
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="flowUE_mean",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Mean traffic flow (veh/hour)}$'},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,#0.2,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'UEflow_MEAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by mean PO flows
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="flowPO_mean",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Mean traffic flow (veh/hour)}$'},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'POflow_MEAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = 0
vMax = max(max(shps['flowUE_median']), max(shps['flowPO_median']))

In [ ]:
# Plot LSOAs coloured by median UE flows
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="flowUE_mean",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Median traffic flow (veh/hour)}$'},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,#0.2,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'UEflow_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by median PO flows
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="flowPO_mean",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Median traffic flow (veh/hour)}$'},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'POflow_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMax = max(abs(min(shps.flowDiff_median)), abs(max(shps.flowDiff_median)))
vMin = -vMax

In [ ]:
# Plot LSOAs coloured by median flow diff
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="flowDiff_median",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Change in median flow (veh/hour)}$'},#, "ticks": [0]},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=brbg,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'flowDiff_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = 0
vMax = max(max(shps.no2UE_median), max(shps.no2PO_median))

In [ ]:
# Plot LSOAs coloured by median UE NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="no2UE_median",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Median NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'UE_NO2_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by median PO NO2
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="no2PO_median",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Median NO}_2 \textrm{ concentration (\SI{}{\ug}/m} ^3 \textrm{)}$"},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=oranges,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'PO_NO2_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMax = max(abs(min(shps.no2Diff_median)), abs(max(shps.no2Diff_median)))
vMin = -vMax

In [ ]:
# Plot LSOAs coloured by median NO2 diff
fig, ax = plt.subplots()
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{siunitx}')
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="no2Diff_median",
                categorical=False,
                legend=True,
                legend_kwds={"label": r"$\textrm{Change in median NO}_2 \textrm{ (\SI{}{\ug}/m} ^3 \textrm{)}$"},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=brbg,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'NO2Diff_MEDIAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
cmap_reversed = mpl.colormaps['Pastel2']

In [ ]:
# plot by categorical median no2 diff

fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="no2Diff_median_cat",
                categorical=True,
                legend=True,
                legend_kwds={"bbox_to_anchor": (0.375, 0.3)},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=cmap_reversed,
                ax=ax,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'NO2Diff_cat_MEDIAN.png', transparent=False, bbox_inches="tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by total intervention index
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="intInd",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Sum of intervention index}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'intInd_TOT.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
shps = shps.assign(intIntSimp = np.multiply((shps['flowUE'] - min(shps['flowUE']))/(max(shps['flowUE']) - min(shps['flowUE'])),
                                            (shps['no2UE'] - min(shps['no2UE']))/(max(shps['no2UE']) - min(shps['no2UE']))))

In [ ]:
# Plot LSOAs coloured by simplified intervention index
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="intIntSimp",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Sum of intervention index}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'intInd_SIMP.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by mean intervention index
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="intIndMean",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Mean intervention index}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'intInd_MEAN.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by company count
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="company_count",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Count}$'},#, "ticks": [0]},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                vmin=0,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'company_count.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot LSOAs coloured by population
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
shps.plot(column="popn",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Count}$'},#, "ticks": [0]},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
shps.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'population_count.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = min(msoas['Income_Total_2020'])
vMax = max(msoas['Income_Total_2020'])
print(vMin,vMax)

In [ ]:
# Plot MSOAs coloured by total annual income (£)
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
msoas.plot(column="Income_Total_2020",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Annual income (£)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
msoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'MSOA_Income_Total_2020.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
vMin = min(msoas['Income_Net_2020'])
vMax = max(msoas['Income_Net_2020'])
print(vMin,vMax)

In [ ]:
# Plot MSOAs coloured by net annual income (£)
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
msoas.plot(column="Income_Net_2020",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Annual income (£)}$'},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=blues,
                ax=ax,
                zorder=1,
                vmin=vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
msoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'MSOA_Income_Net_2020.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# mean change in travel time for all roads (PO compared to UE)
print(np.mean(TT_PO - TT_UE))

In [ ]:
# travel times per km (i.e. km/hour)
TT_UE = np.divide(km, np.array(TT_UE).T).T
TT_PO = np.divide(km, np.array(TT_PO).T).T

In [ ]:
print(np.mean(TT_UE), np.mean(TT_PO), np.median(TT_UE), np.median(TT_PO), np.max(TT_UE), np.max(TT_PO))

In [ ]:
# which roads exceed travel time (PO compared to UE)
PO_exceeds_UE_TT = np.where(TT_PO > TT_UE)[0]
# how many exceed, total number of roads, % of exceeds to total
print(len(PO_exceeds_UE_TT), len(TT_PO), (len(PO_exceeds_UE_TT)/len(TT_PO))*100)

In [ ]:
# mean % increase in PO travel time for those exceeding UE case
np.mean((TT_PO[PO_exceeds_UE_TT] - TT_UE[PO_exceeds_UE_TT])*100 / (TT_UE[PO_exceeds_UE_TT]))

In [ ]:
PO_exceeds_max_UE_TT = np.where(TT_PO > np.max(TT_UE))[0]
print(len(PO_exceeds_max_UE_TT), len(TT_PO), (len(PO_exceeds_max_UE_TT)/len(TT_PO))*100)

In [ ]:
pd.DataFrame(TT_UE).to_csv(mainDir + 'MATLAB Outputs/TT_UE_km.csv')
pd.DataFrame(TT_PO).to_csv(mainDir + 'MATLAB Outputs/TT_PO_km.csv')

no2UE_km.to_csv(mainDir + 'MATLAB Outputs/no2UE_km.csv')
no2PO_km.to_csv(mainDir + 'MATLAB Outputs/no2PO_km.csv')

In [ ]:
NO2_per_km_reduced = np.where(no2PO_km < no2UE_km)[0]
print(len(NO2_per_km_reduced), len(G_edges), len(NO2_per_km_reduced)*100/len(G_edges))

In [ ]:
# Plot travel times histogram
fig, ax = plt.subplots()

plt.hist(TT_UE, color="black", zorder=0, label="Current", density=True, alpha=0.75, bins=50)
plt.hist(TT_PO, color="#5080bb", zorder=0, label="Pollution-optimal", density=True, alpha=0.75, bins=50)
plt.rc('font')
plt.rc('axes')
plt.legend()
#plt.grid(zorder=1)
plt.xlabel(r'$\textrm{Travel time (km/hour)}$')#plt.xlabel('')
plt.ylabel(r'$\textrm{Number of roads}$')

# ticks = [0, 0.015]
# ax.set_xticks(ticks)
# dic = {0.015 : "Max"}
# labels = [ticks[i] if t not in dic.keys() else dic[t] for i,t in enumerate(ticks)]
# ax.set_xticklabels(labels)

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'Travel_Times_Hist.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot NO2 histogram
fig, ax = plt.subplots()

plt.hist(no2UE, color="black", zorder=0, label="Current", density=True, alpha=0.75, bins=50)
plt.hist(no2PO, color="#5080bb", zorder=0, label="Pollution-optimal", density=True, alpha=0.75, bins=50)
plt.rc('font')
plt.rc('axes')
plt.legend()
#plt.grid(zorder=1)
plt.xlabel('')#plt.xlabel(r'$\textrm{NO}_2 \textrm{ concentration (scaled)}$')
plt.ylabel(r'$\textrm{Number of roads}$')

# ticks = [0, 0.0025]
# ax.set_xticks(ticks)
# dic = {0.0025 : "Max"}
# labels = [ticks[i] if t not in dic.keys() else dic[t] for i,t in enumerate(ticks)]
# ax.set_xticklabels(labels)

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'NO2_Hist_noX.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# Plot NO2 histogram per km
fig, ax = plt.subplots()

plt.hist(no2UE_km, color="black", zorder=0, label="Current", density=True, alpha=0.75, bins=50)
plt.hist(no2PO_km, color="#5080bb", zorder=0, label="Pollution-optimal", density=True, alpha=0.75, bins=50)
plt.rc('font')
plt.rc('axes')
plt.legend()
#plt.grid(zorder=1)
plt.xlabel(r'$\textrm{NO}_2 \textrm{ concentration (ppb/km)}$')#plt.xlabel('')
plt.ylabel(r'$\textrm{Number of roads}$')

# ticks = [0, 0.012]
# ax.set_xticks(ticks)
# dic = {0.012 : "Max"}
# labels = [ticks[i] if t not in dic.keys() else dic[t] for i,t in enumerate(ticks)]
# ax.set_xticklabels(labels)

#fig.set_size_inches(w,h)

#plt.savefig(imgOutDir + 'NO2_Hist_KM.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
# ROAD CAPACITIES AND CRITICAL DENSITIES #

In [ ]:
cap = G_edges['capacity']
crit = G_edges['criticalDensity']

In [ ]:
exceedsCapUE = np.where(flowUE.values > cap.values)[1]
exceedsCapPO = np.where(flowPO.values > cap.values)[1]

print(len(exceedsCapUE), len(exceedsCapPO), exceedsCapUE, exceedsCapPO)

In [ ]:
exceedsCritUE = np.where(densityUE.values > crit.values)[1]
exceedsCritPO = np.where(densityPO.values > crit.values)[1]

print(len(exceedsCritUE), len(exceedsCritPO), exceedsCritUE, exceedsCritPO)

In [ ]:
display(msoas.head())

In [ ]:
lsoa_msoa_lookup = lsoa_msoa_lookup.loc[lsoa_msoa_lookup['MSOA11CD'].isin(msoas['MSOA11CD'])].reset_index(drop=True)
lsoa_msoa_lookup = lsoa_msoa_lookup[['LSOA11CD','MSOA11CD']].drop_duplicates(ignore_index=True).reset_index(drop=True)
lsoa_msoa_lookup = lsoa_msoa_lookup.merge(shps[['LSOA11CD','flowUE','flowPO','no2UE','no2PO']], how='left',
                                          left_on='LSOA11CD', right_on='LSOA11CD')
lsoa_msoa_lookup = lsoa_msoa_lookup.fillna(0)

In [ ]:
msoaGroups = lsoa_msoa_lookup.groupby('MSOA11CD').sum(['flowUE','flowPO','no2UE','no2PO'])

In [ ]:
msoas = msoas.merge(msoaGroups, how='left', left_on='MSOA11CD', right_on='MSOA11CD')

In [ ]:
#msoas.to_csv(mainDir + 'Python Outputs/New Results/MSOAs_with_Income_2020.csv')

In [ ]:
msoaCorrDf = msoas[['flowUE','flowPO','no2UE','no2PO','Income_Total_2020','Income_Net_2020']]
msoaCorrDf = msoaCorrDf.loc[~np.isnan(msoaCorrDf['flowUE'])]
# rescale the incomes
msoaCorrDf['Income_Total_2020'] = msoaCorrDf['Income_Total_2020']/np.nansum(msoaCorrDf['Income_Total_2020'])
msoaCorrDf['Income_Net_2020'] = msoaCorrDf['Income_Net_2020']/np.nansum(msoaCorrDf['Income_Net_2020'])
# create correlation matrix
msoaCorrDf = msoaCorrDf.corr()
display(msoaCorrDf)

In [ ]:
#msoaCorrDf.to_csv(mainDir + 'Python Outputs/New Results/MSOA_Correlation_Matrix_NoIntra_Unscaled_Linestring.csv')

In [ ]:
0.04929184597384385 - -0.0046002794606114965

In [ ]:
vMin = 0
vMax = max(max(msoas['flowUE']),max(msoas['flowPO']))

In [ ]:
# Plot MSOAs coloured by UE flows
fig, ax = plt.subplots()
sheff.plot(ax=ax, color="#E7E7E7", zorder=0) #2C2C2C
msoas.plot(column="flowUE",
                categorical=False,
                legend=True,
                legend_kwds={"label": r'$\textrm{Total traffic flow (veh/hour)}$'},#, "ticks": []},
                missing_kwds={"color": "black", "label": r'$\textrm{Missing data}$'},
                cmap=purples,
                ax=ax,
                zorder=1,
                vmin = vMin,
                vmax=vMax)
ox.plot_graph(
    G,
    ax=ax,
    show=False,
    close=False,
    bgcolor="w",
    edge_color="k",
    edge_linewidth=thicknesses,#0.2,
    node_size=0
)
sheff.boundary.plot(color="#4A4949", linewidth=0.3, ax=ax, zorder=2)
msoas.boundary.plot(color="darkgrey", linewidth=0.1, ax=ax, zorder=2)

ax.set_xlim([min(xy.x), max(xy.x)])
ax.set_ylim([min(xy.y), max(xy.y)])

#plt.savefig(imgOutDir + 'MSOA_UEflow_TOT.png', transparent=False, bbox_inches = "tight")
plt.show()

In [ ]:
shps = pd.merge(shps, lsoa_msoa_lookup, left_on='LSOA11CD', right_on='LSOA11CD', how='left')

In [ ]:
sum(shps.no2Diff_median_cat == 'Reduced')

In [ ]:
sum(shps.no2Diff_median_cat == 'Increased')

In [ ]:
sum(shps.no2Diff_median_cat == 'No change')

In [ ]:
sum(shps.no2Diff_median_cat == 'Reduced')/len(shps)*100

In [ ]:
sum(shps.no2Diff_median_cat == 'Increased')/len(shps)*100

In [ ]:
noNanFlowUE = np.array(shps.flowUE_median[~np.isnan(shps.flowUE_median)])
noNanFlowPO = np.array(shps.flowPO_median[~np.isnan(shps.flowPO_median)])

noNanNo2UE = np.array(shps.no2UE_median[~np.isnan(shps.no2UE_median)])
noNanNo2PO = np.array(shps.no2PO_median[~np.isnan(shps.no2PO_median)])

In [ ]:
sortFlowUE = shps.sort_values(by=['flowUE_median'], ascending=False)
sortFlowPO = shps.sort_values(by=['flowPO_median'], ascending=False)

sortNo2UE = shps.sort_values(by=['no2UE_median'], ascending=False)
sortNo2PO = shps.sort_values(by=['no2PO_median'], ascending=False)

In [ ]:
sortCompany = shps.sort_values(by=['company_count'], ascending=False)
sortPopn = shps.sort_values(by=['popn'], ascending=False)

In [ ]:
sortIntInd = shps.sort_values(by=['intInd'], ascending=False)

In [ ]:
sortDemand = shps.sort_values(by=['Inter_LSOA'], ascending=False)

In [ ]:
print(np.mean(shps['company_count']), np.median(shps['company_count']), min(shps['company_count']), max(shps['company_count']))

In [ ]:
print(np.mean(shps['popn']), np.median(shps['popn']), min(shps['popn']), max(shps['popn']))

In [ ]:
print(len(shps), 10*100/len(shps))

In [ ]:
sum(np.array(sortFlowUE.iloc[0:10]['flowUE']))/sum(noNanFlowUE)*100

In [ ]:
# Burngreave & Grimesthorpe
bg_and_gt = shps[shps['msoa21cd']=='E02001632']

In [ ]:
# % flows in Burngreave & Grimesthorpe
np.sum(bg_and_gt['flowUE'])*100/np.sum(shps['flowUE'])

In [ ]:
# Tinsley & Carbrook
t_and_c = shps[shps['msoa21cd']=='E02001628']

In [ ]:
# % flows in Tinsley & Carbrook
np.sum(t_and_c['flowUE'])*100/np.sum(shps['flowUE'])

In [ ]:
sum(np.array(sortFlowUE.iloc[0:2]['flowUE']))/sum(np.array(bg_and_gt['flowUE']))*100

In [ ]:
sortIntInd.iloc[0:20][['intInd','LSOA11CD','company_count','popn']]

In [ ]:
devGreenLSOAs = ['E01008114','E01008115','E01033265','E01034844']
devTable = shps.loc[shps['LSOA11CD'].isin(devGreenLSOAs)]
display(devTable[['LSOA11CD','flowUE','flowPO','no2UE','no2PO']])

In [ ]:
sortFlowUE.iloc[0:10][['flowUE_median','LSOA11CD','company_count','popn','flowDiff_median']]

In [ ]:
sum(sortFlowUE.iloc[1:10]['flowUE_median'])*100/sum(sortFlowUE['flowUE_median'])

In [ ]:
sum(sortFlowUE.iloc[0:10]['flowDiff_median'])*100/sum(sortFlowUE.iloc[0:10]['flowUE_median'])

In [ ]:
sum(sortFlowUE['flowDiff_median'])*100/sum(sortFlowUE['flowUE_median'])

In [ ]:
sum(sortFlowUE.iloc[1:9]['flowDiff_median'])*100/sum(sortFlowUE.iloc[1:9]['flowUE_median'])

In [ ]:
# city centre
np.sum(sortFlowUE.iloc[[2,3,4]]['flowUE_median'])*100/np.sum(sortFlowUE['flowUE_median'])

In [ ]:
# Fulwood & Lodge Moor
np.sum(sortFlowUE.iloc[0]['flowUE_median'])*100/np.sum(sortFlowUE['flowUE_median'])

In [ ]:
# Oughtibridge & Bradfield
np.sum(sortFlowUE.iloc[1]['flowUE'])*100/np.sum(sortFlowUE['flowUE'])

In [ ]:
# All top 10
np.sum(sortFlowUE.iloc[0:10]['flowUE'])*100/np.sum(sortFlowUE['flowUE'])

In [ ]:
np.sum(sortFlowUE.iloc[8]['flowUE'])*100/np.sum(sortFlowUE['flowUE'])

In [ ]:
len(noNanFlowUE)

In [ ]:
sum(sortFlowUE.iloc[0:10]['flowUE'])/sum(noNanFlowUE)*100

In [ ]:
display(shps.loc[shps['flowDiff_median'] == max(shps['flowDiff_median']), ['LSOA11CD','flowUE_median','flowPO_median','flowDiff_median','company_count','popn']])

In [ ]:
G_edges.iloc[np.where(shps['flowDiff_median'] == max(shps['flowDiff_median']))[0][0]]

In [ ]:
print(shps.loc[shps['flowDiff_median'] == max(shps['flowDiff_median']), 'flowPO_median'], np.median(shps['flowPO_median']), np.mean(shps['flowPO_median']))

In [ ]:
print(shps.loc[shps['flowDiff_median'] == max(shps['flowDiff_median']), 'no2PO_median'])

In [ ]:
sortFlowPO.iloc[0:10][['flowPO_median','LSOA11CD','flowDiff_median','company_count','popn']]

In [ ]:
sum(sortFlowPO.iloc[0:10]['flowPO'])/sum(noNanFlowPO)*100

In [ ]:
np.array(sortFlowPO.iloc[0:10]['flowPO'])/sum(noNanFlowPO)*100

In [ ]:
sum(np.array(sortFlowPO.iloc[0:4]['flowPO']))/sum(noNanFlowPO)*100

In [ ]:
sortNo2UE.iloc[0:10]['LSOA11CD'].reset_index(drop=True) == sortFlowUE.iloc[0:10]['LSOA11CD'].reset_index(drop=True)

In [ ]:
100-sum(sortNo2UE['no2PO_median'])*100/sum(sortNo2UE['no2UE_median'])

In [ ]:
display(sortNo2UE.iloc[0:10][['no2UE_median','LSOA11CD','company_count','popn','no2Diff_median']])

In [ ]:
# city centre
np.sum(sortNo2UE.iloc[[1,2,3]]['no2UE'])*100/np.sum(sortNo2UE['no2UE'])

In [ ]:
# Park Hill
np.sum(sortNo2UE.iloc[9]['no2UE'])*100/np.sum(sortNo2UE['no2UE'])

In [ ]:
# All Cathedral & Kelham
cat_and_kel = shps[shps['msoa21cd']=='E02006843']

In [ ]:
np.sum(cat_and_kel['no2UE'])

In [ ]:
sum(sortNo2UE.iloc[0:10]['no2UE'])/sum(noNanNo2UE)*100

In [ ]:
topTenFlowUE = sortFlowUE.iloc[0:10]
topTenFlowPO = sortFlowPO.iloc[0:10]

topTenNo2UE = sortNo2UE.iloc[0:10]
topTenNo2PO = sortNo2PO.iloc[0:10]

In [ ]:
display(topTenFlowUE.loc[~(topTenFlowUE['msoa21cd'].isin(topTenFlowPO['msoa21cd'])), ['no2UE','LSOA11CD','msoa21cd','company_count','popn']])

In [ ]:
display(topTenNo2UE.loc[~(topTenNo2UE['msoa21cd'].isin(topTenNo2PO['msoa21cd'])), ['no2UE','LSOA11CD','msoa21cd','company_count','popn']])

In [ ]:
shps.loc[shps['no2UE'] == max(shps['no2UE']), 'LSOA11CD']

In [ ]:
sortNo2PO.iloc[0:10][['no2PO_median','LSOA11CD']]

In [ ]:
sum(sortNo2UE.iloc[[0,2]]['no2UE'])/sum(cat_and_kel['no2UE'])*100

In [ ]:
sum(sortNo2UE.iloc[[4,9]]['no2UE'])/sum(t_and_c['no2UE'])*100

In [ ]:
sum(np.array(sortNo2UE.iloc[[3,5]]['no2UE']))/sum(np.array(shps.loc[~np.isnan(shps['no2UE']), 'no2UE']))*100

In [ ]:
sum(sortNo2UE.iloc[[3,5]]['no2UE'])

In [ ]:
sum(t_and_c['no2UE'])

In [ ]:
sum(np.array(sortNo2UE.iloc[0:2]['no2UE']))/sum(np.array(bg_and_gt['no2UE']))*100

In [ ]:
sum(np.array(sortNo2UE.iloc[3:5]['no2UE']))/sum(np.array(shps.loc[~np.isnan(shps['no2UE']), 'no2UE']))*100

In [ ]:
sum(np.array(sortNo2UE.iloc[3:5]['no2UE']))/sum(np.array(t_and_c['no2UE']))*100

In [ ]:
sortNo2UE.iloc[0:3]['no2UE']

In [ ]:
100 - np.sum(no2PO)*100/np.sum(no2UE)

In [ ]:
display(shps[['no2UE_area','no2PO_area']])

In [ ]:
print(max(shps['no2UE_area']), max(shps['no2PO_area']))

In [ ]:
msoas[['Income_Net_2020','no2UE']].head()

In [ ]:
topN = len(sortIntInd)
rankIntInd = sortIntInd.iloc[0:topN]['LSOA11CD']
rankDemand = sortDemand.iloc[0:topN]['LSOA11CD']
rankNO2UE = sortNo2UE.iloc[0:topN]['LSOA11CD']
rankNO2PO = sortNo2PO.iloc[0:topN]['LSOA11CD']
rankPopn = sortPopn.iloc[0:topN]['LSOA11CD']
rankCompany = sortCompany.iloc[0:topN]['LSOA11CD']

sortIncome_MSOA = msoas.sort_values(by=['Income_Net_2020'], ascending=False)
sortNO2UE_MSOA = msoas.sort_values(by=['no2UE'], ascending=False)
sortNO2PO_MSOA = msoas.sort_values(by=['no2PO'], ascending=False)

topN_MSOA = 5
rankIncome_MSOA = sortIncome_MSOA.iloc[0:topN_MSOA]['MSOA11CD']
rankNO2UE_MSOA = sortNO2UE_MSOA.iloc[0:topN_MSOA]['MSOA11CD']
rankNO2PO_MSOA = sortNO2PO_MSOA.iloc[0:topN_MSOA]['MSOA11CD']

In [ ]:
scipy.stats.kendalltau(rankIntInd, rankCompany, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankIntInd, rankPopn, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankIntInd, rankDemand, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankPopn, rankDemand, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankPopn, rankNO2UE, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankPopn, rankNO2PO, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankPopn, rankNO2UE, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankIncome_MSOA, rankNO2UE_MSOA, alternative='greater')

In [ ]:
scipy.stats.kendalltau(rankIncome_MSOA, rankNO2PO_MSOA, alternative='greater')

In [ ]:
np.corrcoef(msoas['Income_Net_2020'], msoas['no2UE'])